In [1]:

# ================================================================
# JEEVAN LINK - ETA MODEL TESTING
# Model Loading + Scenario Testing + Evaluation
# ================================================================

import pandas as pd
import numpy as np
import json
import joblib

from pathlib import Path
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ================================================================
# 1. PATHS
# ================================================================

PROJECT_DIR = Path(r"C:\Users\ishit\jeevan_link_ml")

MODEL_DIR = PROJECT_DIR / "Models" / "ETA_Model"

MODEL_PATH = MODEL_DIR / "eta_model.pkl"
FEATURE_PATH = MODEL_DIR / "eta_feature_names.pkl"
TARGET_PATH = MODEL_DIR / "eta_target_name.pkl"
PREDICTIONS_PATH = MODEL_DIR / "eta_test_predictions.csv"

print("=" * 65)
print("JEEVAN LINK - ETA MODEL TESTING")
print("=" * 65)

# ================================================================
# 2. LOAD TRAINED MODEL
# ================================================================

required_files = [
    MODEL_PATH,
    FEATURE_PATH,
    TARGET_PATH,
    PREDICTIONS_PATH
]

for file in required_files:
    if not file.exists():
        raise FileNotFoundError(f"Required file missing: {file}")

eta_model = joblib.load(MODEL_PATH)
feature_names = joblib.load(FEATURE_PATH)
target_name = joblib.load(TARGET_PATH)

print("\nMODEL LOADED SUCCESSFULLY")
print("Target:", target_name)
print("Number of features:", len(feature_names))

# ================================================================
# 3. LOAD HELD-OUT TEST PREDICTIONS
# ================================================================

test_df = pd.read_csv(PREDICTIONS_PATH)

actual = test_df["actual_travel_minutes"]
predicted = test_df["predicted_travel_minutes"]

# ================================================================
# 4. RECHECK MODEL PERFORMANCE
# ================================================================

mae = mean_absolute_error(actual, predicted)

rmse = np.sqrt(
    mean_squared_error(actual, predicted)
)

r2 = r2_score(actual, predicted)

print("\n" + "=" * 65)
print("HELD-OUT TEST PERFORMANCE")
print("=" * 65)

print("MAE:", round(mae, 4), "minutes")
print("RMSE:", round(rmse, 4), "minutes")
print("R2 Score:", round(r2, 4))
print("R2 Percentage:", round(r2 * 100, 2), "%")

# ================================================================
# 5. PREDICTION ERROR ANALYSIS
# ================================================================

test_df["absolute_error"] = np.abs(
    test_df["actual_travel_minutes"]
    - test_df["predicted_travel_minutes"]
)

print("\nERROR ANALYSIS")

print(
    "Minimum error:",
    round(test_df["absolute_error"].min(), 2),
    "minutes"
)

print(
    "Maximum error:",
    round(test_df["absolute_error"].max(), 2),
    "minutes"
)

print(
    "Average error:",
    round(test_df["absolute_error"].mean(), 2),
    "minutes"
)

# ================================================================
# 6. TEST SAMPLE PREDICTIONS
# ================================================================

print("\n" + "=" * 65)
print("SAMPLE TEST PREDICTIONS")
print("=" * 65)

print(
    test_df[
        [
            "actual_travel_minutes",
            "predicted_travel_minutes",
            "absolute_error"
        ]
    ].head(15).round(2).to_string(index=False)
)

# ================================================================
# 7. SCENARIO TESTING
# ================================================================

# Each scenario is a complete input row.
# Start with actual held-out test records and modify selected
# features. This guarantees the required feature structure.

# These are demonstration scenarios, not independent real-world data.

test_scenarios = []

# Scenario 1: Short route
scenario = test_df.index[0]
test_scenarios.append(("Short route", scenario))

# Scenario 2: Medium route
scenario = test_df.index[len(test_df) // 3]
test_scenarios.append(("Medium route", scenario))

# Scenario 3: Long route
scenario = test_df.index[-1]
test_scenarios.append(("Long route", scenario))

# Scenario 4: Critical urgency
test_scenarios.append(("Critical urgency", test_df.index[10]))

# Scenario 5: High urgency
test_scenarios.append(("High urgency", test_df.index[20]))

# Scenario 6: Medium urgency
test_scenarios.append(("Medium urgency", test_df.index[30]))

# Scenario 7: Low urgency
test_scenarios.append(("Low urgency", test_df.index[40]))

scenario_results = []

# Load original feature dataset selected during training
print("\nFor scenario testing, please select the SAME ETA dataset used for training.")

from tkinter import Tk, filedialog

root = Tk()
root.withdraw()
root.attributes("-topmost", True)

selected_file = filedialog.askopenfilename(
    title="Select the original ETA training dataset",
    filetypes=[
        ("CSV and Excel files", "*.csv *.xls *.xlsx"),
        ("All files", "*.*")
    ]
)

root.destroy()

if not selected_file:
    raise FileNotFoundError("No ETA dataset selected.")

DATA_PATH = Path(selected_file)

try:
    original_df = pd.read_csv(DATA_PATH)
except (UnicodeDecodeError, pd.errors.ParserError):
    original_df = pd.read_excel(DATA_PATH)

# Reproduce the exact split used in training.
from sklearn.model_selection import train_test_split

X_original = original_df.drop(
    columns=[target_name, "request_id"],
    errors="ignore"
)

y_original = original_df[target_name]

X_train, X_temp, y_train, y_temp = train_test_split(
    X_original,
    y_original,
    test_size=0.30,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

# Use test samples as realistic starting points
scenario_rows = [
    ("Short route", 0),
    ("Medium route", len(X_test) // 3),
    ("Long route", len(X_test) - 1),
    ("Critical urgency", 10),
    ("High urgency", 20),
    ("Medium urgency", 30),
    ("Low urgency", 40)
]

for scenario_name, row_index in scenario_rows:

    sample = X_test.iloc[[row_index]].copy()

    # Change urgency for the urgency-specific scenarios.
    if scenario_name in [
        "Critical urgency",
        "High urgency",
        "Medium urgency",
        "Low urgency"
    ] and "urgency_label" in sample.columns:

        sample["urgency_label"] = scenario_name.split()[0]

    prediction = eta_model.predict(sample)[0]

    scenario_results.append({
        "scenario": scenario_name,
        "predicted_eta_minutes": round(float(prediction), 2)
    })

scenario_df = pd.DataFrame(scenario_results)

print("\n" + "=" * 65)
print("SCENARIO TEST RESULTS")
print("=" * 65)

print(scenario_df.to_string(index=False))

scenario_df.to_csv(
    MODEL_DIR / "eta_scenario_test_results.csv",
    index=False
)

# ================================================================
# 8. SAVE FINAL TEST REPORT
# ================================================================

testing_report = {
    "model": "ETA Regression Model",
    "algorithm": "Hist Gradient Boosting",
    "test_records": int(len(test_df)),
    "MAE_minutes": float(mae),
    "RMSE_minutes": float(rmse),
    "R2_score": float(r2),
    "R2_percentage": float(r2 * 100),
    "average_absolute_error_minutes": float(
        test_df["absolute_error"].mean()
    )
}

with open(
    MODEL_DIR / "eta_testing_report.json",
    "w"
) as file:
    json.dump(testing_report, file, indent=4)

# ================================================================
# 9. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 65)
print("ETA MODEL TESTING COMPLETED")
print("=" * 65)

print("\nFinal Metrics:")
print("MAE:", round(mae, 3), "minutes")
print("RMSE:", round(rmse, 3), "minutes")
print("R2:", round(r2 * 100, 2), "%")

print("\nSaved testing files:")
print("- eta_scenario_test_results.csv")
print("- eta_testing_report.json")

print("\nModel folder:")
print(MODEL_DIR)

print("\nETA MODEL TESTING SUCCESSFUL!")


JEEVAN LINK - ETA MODEL TESTING

MODEL LOADED SUCCESSFULLY
Target: actual_travel_minutes
Number of features: 27

HELD-OUT TEST PERFORMANCE
MAE: 1.6508 minutes
RMSE: 2.0957 minutes
R2 Score: 0.9744
R2 Percentage: 97.44 %

ERROR ANALYSIS
Minimum error: 0.0 minutes
Maximum error: 6.96 minutes
Average error: 1.65 minutes

SAMPLE TEST PREDICTIONS
 actual_travel_minutes  predicted_travel_minutes  absolute_error
                   7.1                     10.27            3.17
                  31.5                     32.93            1.43
                  24.1                     24.81            0.71
                  33.6                     31.43            2.17
                  20.5                     21.14            0.64
                  28.3                     31.74            3.44
                  43.3                     42.08            1.22
                  39.0                     38.41            0.59
                  50.6                     49.91            0.69
      